In [ ]:
import pandas as pd
import re
from typing import Dict, Tuple

In [ ]:
DATASET_RULES = {
    'production': {
        'patterns': [
            r'batch_production_\d+.*',
            r'\d+_myekyc_.*',
        ],
        'weights': {'apcer': -1.0, 'bpcer': -1.0},
    },
    'datacollection': {
        'patterns': [
            r'batch_datacollection_\d+.*',
        ],
        'weights': {'apcer': -1.0, 'bpcer': -1.0},
    },
    'issue': {
        'patterns': [
            r'batch_issue_\d+.*',
        ],
        'weights': {'apcer': -0.0769, 'bpcer': 0},
    },
    'test_plan': {
        'patterns': [
            r'.*test_plan_subject.*',
            r'grayscale_print.*',
            r'colorprint.*',
            r'replay_.*',
        ],
        'weights': {'apcer': -1.0, 'bpcer': -1.0},
    },
}

# Special cases with exact match
SPECIAL_CASES = {
    'batch_issue_set/index_annotation_mykadfront_background.csv': {
        'weights': {'apcer': 0, 'bpcer': -0.0769},
    },
    'batch_issue_20240412_partner_both_general/index_annotation_mykadfront': {
        'weights': {'apcer': -0.0769, 'bpcer': -0.0769},
    }
}

def classify_dataset(dataset_name: str) -> Tuple[str, Dict[str, float]]:
    """Classify dataset and return (category, weights)."""
    # Check special cases first
    if dataset_name in SPECIAL_CASES:
        return 'special', SPECIAL_CASES[dataset_name]['weights']

    # Check against pattern rules
    for category, rules in DATASET_RULES.items():
        for pattern in rules['patterns']:
            if re.match(pattern, dataset_name, re.IGNORECASE):
                return category, rules['weights']

    # Default
    return 'unknown', {'apcer': -1.0, 'bpcer': -1.0}

In [ ]:
def classify_dataset(dataset_name: str) -> Tuple[str, Dict[str, float]]:
    """Classify dataset and return (category, weights)."""
    # Check special cases first
    if dataset_name in SPECIAL_CASES:
        return 'special', SPECIAL_CASES[dataset_name]['weights']

    # Check against pattern rules
    for category, rules in DATASET_RULES.items():
        for pattern in rules['patterns']:
            if re.match(pattern, dataset_name, re.IGNORECASE):
                return category, rules['weights']

    # Default
    return 'unknown', {'apcer': -1.0, 'bpcer': -1.0}

In [ ]:
# Load predictions CSV
csv_path = '/home/jingjie/AutoTorch/runs/Exp2_dinov3_convnext_large_v1_512_crop/predictions_full.csv'
df = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded {len(df)} rows")
print(f"Checkpoint columns: {[c for c in df.columns if c.startswith('pred_prob_ckpt')]}")

Loaded 33008 rows
Checkpoint columns: ['pred_prob_ckpt1', 'pred_prob_ckpt2', 'pred_prob_ckpt3', 'pred_prob_ckpt4', 'pred_prob_ckpt5', 'pred_prob_ckpt6', 'pred_prob_ckpt7', 'pred_prob_ckpt8', 'pred_prob_ckpt9', 'pred_prob_ckpt10', 'pred_prob_ckpt11', 'pred_prob_ckpt12', 'pred_prob_ckpt13', 'pred_prob_ckpt14', 'pred_prob_ckpt15', 'pred_prob_ckpt16', 'pred_prob_ckpt17', 'pred_prob_ckpt18', 'pred_prob_ckpt19', 'pred_prob_ckpt20']


In [ ]:
def calculate_metrics(df, prob_col, threshold=0.5, target_col='label'):
    """
    Calculate APCER/BPCER metrics from predictions DataFrame.
    
    Args:
        df: DataFrame with predictions and labels
        prob_col: Column with predicted probabilities
        threshold: Classification threshold (default 0.5)
        target_col: Column with ground truth (1=attack, 0=bonafide)
    
    Returns:
        Dict with TP, FP, TN, FN, apcer, bpcer, acer, accuracy
    """
    y_true = df[target_col].values
    y_pred = (df[prob_col].values > threshold).astype(int)

    TP = int(((y_pred == 1) & (y_true == 1)).sum())
    FP = int(((y_pred == 1) & (y_true == 0)).sum())
    TN = int(((y_pred == 0) & (y_true == 0)).sum())
    FN = int(((y_pred == 0) & (y_true == 1)).sum())

    apcer = FN / (FN + TP) if (FN + TP) > 0 else 0.0
    bpcer = FP / (FP + TN) if (FP + TN) > 0 else 0.0
    acer = (apcer + bpcer) / 2
    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0.0

    return {
        'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
        'apcer': apcer, 'bpcer': bpcer, 'acer': acer, 'accuracy': accuracy
    }

In [ ]:
def calculate_weighted_score(metrics_per_batch):
    """
    Calculate weighted score from per-batch metrics.
    
    Formula: score = Σ((1 - metric) * |weight|) / Σ(|weight|)
    Higher score = better performance.
    """
    score = 0
    total_weight = 0

    for batch_name, metrics in metrics_per_batch.items():
        _, weights = classify_dataset(batch_name)

        for metric_name in ['apcer', 'bpcer']:
            weight = weights.get(metric_name, 0)
            if weight != 0:
                total_weight += abs(weight)
                score += (1 - metrics[metric_name]) * abs(weight)

    return round(score / total_weight, 4) if total_weight > 0 else 0


def evaluate_epoch(df, prob_col, threshold=0.5):
    """Calculate metrics per original_batch_name for one epoch."""
    metrics_per_batch = {}

    for batch_name in df['original_batch_name'].unique():
        batch_df = df[df['original_batch_name'] == batch_name]
        metrics = calculate_metrics(batch_df, prob_col, threshold)
        metrics_per_batch[batch_name] = metrics

    return metrics_per_batch


def generate_leaderboard(df, threshold=0.5):
    """Generate leaderboard comparing all checkpoint columns."""
    ckpt_cols = sorted(
        [(int(re.search(r'ckpt(\d+)$', col).group(1)), col) 
         for col in df.columns if re.match(r'^pred_prob_ckpt\d+$', col)]
    )

    results = []
    for epoch, col in ckpt_cols:
        metrics_per_batch = evaluate_epoch(df, col, threshold)
        weighted_score = calculate_weighted_score(metrics_per_batch)
        overall = calculate_metrics(df, col, threshold)

        results.append({
            'epoch': epoch,
            'weighted_score': weighted_score,
            'overall_apcer': overall['apcer'],
            'overall_bpcer': overall['bpcer'],
            'overall_acer': overall['acer'],
        })
        print(f"Epoch {epoch:>2}: score={weighted_score:.4f}, "
              f"APCER={overall['apcer']*100:.2f}%, BPCER={overall['bpcer']*100:.2f}%")

    return pd.DataFrame(results)

In [ ]:
# Generate leaderboard
leaderboard = generate_leaderboard(df, threshold=0.5)

Epoch  1: score=0.9893, APCER=0.50%, BPCER=5.62%
Epoch  2: score=0.9516, APCER=0.07%, BPCER=29.73%
Epoch  3: score=0.9786, APCER=1.85%, BPCER=1.40%
Epoch  4: score=0.9881, APCER=0.16%, BPCER=9.98%
Epoch  5: score=0.9863, APCER=0.89%, BPCER=2.69%
Epoch  6: score=0.9871, APCER=1.02%, BPCER=1.45%
Epoch  7: score=0.9895, APCER=0.35%, BPCER=5.82%
Epoch  8: score=0.9833, APCER=1.87%, BPCER=2.06%
Epoch  9: score=0.9218, APCER=17.33%, BPCER=0.32%
Epoch 10: score=0.9909, APCER=0.77%, BPCER=3.64%
Epoch 11: score=0.9698, APCER=4.00%, BPCER=1.19%
Epoch 12: score=0.9901, APCER=0.39%, BPCER=6.47%
Epoch 13: score=0.9872, APCER=0.91%, BPCER=4.72%
Epoch 14: score=0.9901, APCER=0.60%, BPCER=5.30%
Epoch 15: score=0.9902, APCER=0.56%, BPCER=5.06%
Epoch 16: score=0.9896, APCER=0.40%, BPCER=6.69%
Epoch 17: score=0.9837, APCER=1.22%, BPCER=1.81%
Epoch 18: score=0.9776, APCER=2.61%, BPCER=5.13%
Epoch 19: score=0.9875, APCER=0.88%, BPCER=3.86%
Epoch 20: score=0.9838, APCER=0.94%, BPCER=3.12%


In [ ]:
# Display top epochs sorted by weighted score
leaderboard_sorted = leaderboard.sort_values('weighted_score', ascending=False)
print("Top 10 epochs by weighted score:")
leaderboard_sorted.head(10)

Top 10 epochs by weighted score:


,epoch,weighted_score,overall_apcer,overall_bpcer,overall_acer
9,10,0.9909,0.007722,0.036402,0.022062
14,15,0.9902,0.005623,0.050587,0.028105
11,12,0.9901,0.003898,0.064721,0.034310
13,14,0.9901,0.005997,0.053028,0.029513
15,16,0.9896,0.004048,0.066856,0.035452
6,7,0.9895,0.003524,0.058163,0.030843
0,1,0.9893,0.005023,0.056180,0.030601
3,4,0.9881,0.001574,0.099751,0.050663
18,19,0.9875,0.008846,0.038589,0.023717
12,13,0.9872,0.009071,0.047181,0.028126


In [ ]:
# Best epoch summary
best = leaderboard_sorted.iloc[0]
print(f"Best epoch: {int(best['epoch'])}")
print(f"  Weighted Score: {best['weighted_score']:.4f}")
print(f"  Overall APCER:  {best['overall_apcer']*100:.2f}%")
print(f"  Overall BPCER:  {best['overall_bpcer']*100:.2f}%")
print(f"  Overall ACER:   {best['overall_acer']*100:.2f}%")

# Save leaderboard
output_path = csv_path.replace('.csv', '_leaderboard.csv')
leaderboard_sorted.to_csv(output_path, index=False)
print(f"\nLeaderboard saved to {output_path}")

Best epoch: 10
  Weighted Score: 0.9909
  Overall APCER:  0.77%
  Overall BPCER:  3.64%
  Overall ACER:   2.21%

Leaderboard saved to /home/jingjie/AutoTorch/runs/Exp2_dinov3_convnext_large_v1_512_crop/predictions_full_leaderboard.csv


In [ ]:
# Per-batch breakdown for best epoch
best_epoch = int(best['epoch'])
best_col = f'pred_prob_ckpt{best_epoch}'
metrics_per_batch = evaluate_epoch(df, best_col)

# Create detailed breakdown DataFrame
breakdown = []
for batch_name, metrics in metrics_per_batch.items():
    category, weights = classify_dataset(batch_name)
    breakdown.append({
        'batch': batch_name,
        'category': category,
        'n_genuine': metrics['TN'] + metrics['FP'],
        'n_attack': metrics['TP'] + metrics['FN'],
        'apcer': metrics['apcer'],
        'bpcer': metrics['bpcer'],
        'apcer_weight': weights['apcer'],
        'bpcer_weight': weights['bpcer'],
    })

breakdown_df = pd.DataFrame(breakdown).sort_values('category')

breakdown_output_path = csv_path.replace('.csv', '_leaderboard_breakdown.csv')
breakdown_df.to_csv(breakdown_output_path, index=False)
print(f"\nLeaderboard breakdown saved to {breakdown_output_path}")

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
breakdown_df


Leaderboard breakdown saved to /home/jingjie/AutoTorch/runs/Exp2_dinov3_convnext_large_v1_512_crop/predictions_full_leaderboard_breakdown.csv


,batch,category,n_genuine,n_attack,apcer,bpcer,apcer_weight,bpcer_weight
2,batch_datacollection_20230214_none_recapture/b...,datacollection,500,4000,0.004500,0.000000,-1.0,-1.0
3,batch_datacollection_20231023_wiseai_recapture...,datacollection,160,160,0.000000,0.000000,-1.0,-1.0
4,batch_datacollection_20231219_wiseai_recapture...,datacollection,342,265,0.000000,0.005848,-1.0,-1.0
5,batch_datacollection_20240303_wiseai_recapture...,datacollection,56,56,0.000000,0.000000,-1.0,-1.0
6,batch_datacollection_20240303_wiseai_recapture...,datacollection,56,56,0.000000,0.000000,-1.0,-1.0
...,...,...,...,...,...,...,...,...
40,1.11 Replay Mobile Test Plan/cleaned_filtered_...,unknown,0,1663,0.002405,0.000000,1.0,1.0
41,1.12 Replay Tablet Test Plan/cleaned_filtered_...,unknown,0,178,0.000000,0.000000,1.0,1.0
42,1.1 Genuine with Background Test Plan/cleaned_...,unknown,601,0,0.000000,0.000000,1.0,1.0
34,1.4 Grayscale Print Cutout Test Plan/cleaned_f...,unknown,0,329,0.000000,0.000000,1.0,1.0
